In [11]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import os

# 1. Inicializa o Spark para puxar a amostra da camada Gold
spark = (
    SparkSession.builder.appName("TCC_Modelo_Exploratorio")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print(
    "📥 Lendo a fato_bacen da camada Gold e extraindo uma amostra representativa..."
)
df_fato_spark = spark.read.parquet("../data/processed/gold/fato_bacen")

# Extraindo 1% da base para teste inicial rápido e seguro
df_sample_spark = df_fato_spark.sample(fraction=0.01, seed=42)
df_pd = df_sample_spark.toPandas()

print(f"📊 Amostra carregada no Pandas com {len(df_pd):,} linhas.")

# 2. Carregar e preparar as bases auxiliares (Regiões e Submodalidades Agrupadas)
print(
    "🗺️ Carregando e integrando tabelas de dimensões (Regiões e Grupos de Submodalidades)..."
)
# Caminho exato convertido para o padrão Linux/WSL que você mandou:
base_path = "/home/haroldo/tcc_bacen_scr/data/processed/gold"

# Se os arquivos estiverem direto nessa pasta gold:
df_uf = pd.read_excel(f"{base_path}/dim_uf_completa_sem_acento.xlsx")
df_submod = pd.read_excel(f"{base_path}/submodalidades_agrupadas.xlsx")

# Criando IDs numéricos padronizados via factorize
df_submod["id_grupo_submodalidade"], _ = pd.factorize(
    df_submod["grupo_submodalidade"]
)
df_uf["id_regiao"], _ = pd.factorize(df_uf["regiao"])

# Fazendo o merge (join) das novas chaves na amostra principal
df_pd = df_pd.merge(df_uf[["id_uf", "id_regiao"]], on="id_uf", how="left")
df_pd = df_pd.merge(
    df_submod[["id_submodalidade", "id_grupo_submodalidade"]],
    on="id_submodalidade",
    how="left",
)

# Preenchendo eventuais nulos gerados no merge
df_pd["id_regiao"] = df_pd["id_regiao"].fillna(-1)
df_pd["id_grupo_submodalidade"] = df_pd["id_grupo_submodalidade"].fillna(-1)

# 3. Definindo a Variável Alvo (Target) e as Features (Atributos)
target = "id_perfil_operacao"

features_numericas = [
    "carteira_ativa",
    "carteira_inadimplencia",
    "carteira_vencida",
    "ativo_problematico",
    "numero_de_operacoes",
]

# Incluídas as novas variáveis id_regiao e id_grupo_submodalidade
features_categoricas = [
    "id_uf",
    "id_regiao",
    "id_segmento",
    "id_cliente",
    "id_porte",
    "id_modalidade",
    "id_submodalidade",
    "id_grupo_submodalidade",
    "id_indexador",
]

X = df_pd[features_numericas + features_categoricas]
y = df_pd[target]

# Tratando nulos
X[features_numericas] = X[features_numericas].fillna(0)
X[features_categoricas] = X[features_categoricas].fillna(-1).astype(str)

# Divisão de Treino e Teste (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Construindo Pipeline de Pré-processamento e Modelo (Regressão Logística - GLM)
print("\n⚙️ Treinando Modelo Estatístico (Regressão Logística / GLM Aprimorado)...")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features_numericas),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            features_categoricas,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter=500, class_weight="balanced"),
        ),
    ]
)

model.fit(X_train, y_train)

# 5. Avaliação com Matriz de Confusão e Métricas
y_pred = model.predict(X_test)

print("\n=== MATRIZ DE CONFUSÃO ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

# 6. Analisando os Coeficientes Atualizados
print("\n=== TOP FATORES QUE MAIS PUXAM PARA INADIMPLÊNCIA (COEFICIENTES) ===")
cat_encoder = model.named_steps["preprocessor"].named_transformers_["cat"]
encoded_cat_features = cat_encoder.get_feature_names_out(features_categoricas)
all_features = list(features_numericas) + list(encoded_cat_features)

coefficients = model.named_steps["classifier"].coef_[0]
df_coef = pd.DataFrame({"Feature": all_features, "Coeficiente": coefficients})
df_coef = df_coef.sort_values(by="Coeficiente", ascending=False)

print("--- Maiores Pesos Positivos (Empurram para Inadimplência = 1) ---")
print(df_coef.head(15))

print("\n--- Maiores Pesos Negativos (Empurram para Adimplência = 0) ---")
print(df_coef.tail(15))

📥 Lendo a fato_bacen da camada Gold e extraindo uma amostra representativa...


/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


📊 Amostra carregada no Pandas com 232,881 linhas.
🗺️ Carregando e integrando tabelas de dimensões (Regiões e Grupos de Submodalidades)...

⚙️ Treinando Modelo Estatístico (Regressão Logística / GLM Aprimorado)...

=== MATRIZ DE CONFUSÃO ===
[[23640  3351]
 [ 3605 15981]]

=== RELATÓRIO DE CLASSIFICAÇÃO ===
              precision    recall  f1-score   support

           0       0.87      0.88      0.87     26991
           1       0.83      0.82      0.82     19586

    accuracy                           0.85     46577
   macro avg       0.85      0.85      0.85     46577
weighted avg       0.85      0.85      0.85     46577


=== TOP FATORES QUE MAIS PUXAM PARA INADIMPLÊNCIA (COEFICIENTES) ===
--- Maiores Pesos Positivos (Empurram para Inadimplência = 1) ---
                      Feature  Coeficiente
1      carteira_inadimplencia   104.029840
64           id_modalidade_13     2.638318
85        id_submodalidade_21     2.324044
45               id_cliente_1     1.721651
117       id_s

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pyspark.sql import SparkSession

# =====================================================================
# 1. DEFINIÇÃO DE CAMINHOS ABSOLUTOS
# =====================================================================
gold_path = "/home/haroldo/tcc_bacen_scr/data/processed/gold"
caminho_fato = os.path.join(gold_path, "fato_bacen")
caminho_uf = os.path.join(gold_path, "dim_uf_completa_sem_acento.xlsx")
caminho_submod = os.path.join(gold_path, "submodalidades_agrupadas.xlsx")

# =====================================================================
# 2. INICIALIZAÇÃO DO SPARK E LEITURA (COM DESCARTE RÁPIDO)
# =====================================================================
spark = (
    SparkSession.builder.appName("TCC_Modelo_Fracionario")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("📥 Lendo a fato_bacen via Spark e extraindo amostra...")
df_pd = spark.read.parquet(caminho_fato).sample(fraction=0.02, seed=42).toPandas()
print(f"📊 Amostra carregada com {len(df_pd):,} linhas.")

# 🛑 O PULO DO GATO: Matar o Spark e limpar o lixo da memória imediatamente
spark.stop()
gc.collect()
print("✅ Spark finalizado. RAM liberada com sucesso para a modelagem.")

# =====================================================================
# 3. LEITURA E INTEGRAÇÃO DAS DIMENSÕES
# =====================================================================
print("🗺️ Integrando dimensões...")
df_uf = pd.read_excel(caminho_uf)
df_submod = pd.read_excel(caminho_submod)

df_submod["id_grupo_submodalidade"], _ = pd.factorize(df_submod["grupo_submodalidade"])
df_uf["id_regiao"], _ = pd.factorize(df_uf["regiao"])

df_pd = df_pd.merge(df_uf[["id_uf", "id_regiao"]], on="id_uf", how="left")
df_pd = df_pd.merge(df_submod[["id_submodalidade", "id_grupo_submodalidade"]], on="id_submodalidade", how="left")

# Limpando as tabelas auxiliares da memória
del df_uf, df_submod
gc.collect()

# =====================================================================
# 4. TRATAMENTO (METODOLOGIA)
# =====================================================================
print("⚙️ Aplicando regras econométricas...")

# Imputação do Sigilo
df_pd['numero_de_operacoes'] = np.where(df_pd['numero_de_operacoes'] == -1, 1, df_pd['numero_de_operacoes'])

# Target Fracionário
df_pd['carteira_ativa'] = df_pd['carteira_ativa'].astype(float)
df_pd['carteira_inadimplencia'] = df_pd['carteira_inadimplencia'].astype(float)

df_pd['taxa_inadimplencia'] = np.where(
    df_pd['carteira_ativa'] > 0, 
    df_pd['carteira_inadimplencia'] / df_pd['carteira_ativa'], 
    0.0
)
df_pd['taxa_inadimplencia'] = df_pd['taxa_inadimplencia'].clip(0.0, 1.0)

# =====================================================================
# 5. OTIMIZAÇÃO DE VARIÁVEIS E PESOS
# =====================================================================
features_categoricas = [
    "id_regiao",      # Agrega os estados (5 categorias) em vez de id_uf (27)
    "id_segmento", 
    "id_porte", 
    "id_modalidade", 
    "id_grupo_submodalidade", # Agrupa as submodalidades (15) em vez das 55 brutas
    "id_cnae_ocupacao", 
    "id_origem"
]

df_pd[features_categoricas] = df_pd[features_categoricas].fillna(-1).astype(int).astype(str)

print("⚡ A gerar Dummies nativas do Pandas...")
X = pd.get_dummies(df_pd[features_categoricas], drop_first=True, dtype=int)
X = sm.add_constant(X) 

y = df_pd['taxa_inadimplencia']

# A normalização evita o 'overflow' exponencial sem perder a proporção analítica
pesos_normalizados = df_pd['numero_de_operacoes'] / df_pd['numero_de_operacoes'].mean()

# =====================================================================
# 6. MODELAGEM GLM DIRETA (COM L-BFGS PARA POUPANÇA DE RAM)
# =====================================================================
print("🧠 Treinando o Modelo Logit Fracionário Ponderado...")
modelo_nacional = sm.GLM(
    endog=y,
    exog=X,
    family=sm.families.Binomial(),
    var_weights=pesos_normalizados
).fit(method='lbfgs', maxiter=150) # O L-BFGS processa iterativamente com baixo impacto na RAM

print("\n" + "="*80)
print("🎯 RESULTADOS DO MODELO (CENÁRIO NACIONAL / PERFIL DE MERCADO)")
print("="*80)
print(modelo_nacional.summary())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/10 22:21:29 WARN Utils: Your hostname, DESKTOP-P0CSUOC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/10 22:21:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/10 22:21:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


📥 Lendo a fato_bacen via Spark e extraindo amostra...


/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


📊 Amostra carregada com 464,703 linhas.
✅ Spark finalizado. RAM liberada com sucesso para a modelagem.
🗺️ Integrando dimensões...
⚙️ Aplicando regras econométricas...
⚡ A gerar Dummies nativas do Pandas...
🧠 Treinando o Modelo Logit Fracionário Ponderado...


/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/statsmodels/genmod/generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)



🎯 RESULTADOS DO MODELO (CENÁRIO NACIONAL / PERFIL DE MERCADO)
                 Generalized Linear Model Regression Results                  
Dep. Variable:     taxa_inadimplencia   No. Observations:               464703
Model:                            GLM   Df Residuals:                   464625
Model Family:                Binomial   Df Model:                           77
Link Function:                  Logit   Scale:                          1.0000
Method:                         lbfgs   Log-Likelihood:                -89742.
Date:                Tue, 10 Nov 2026   Deviance:                       48232.
Time:                        22:22:58   Pearson chi2:                 1.09e+05
No. Iterations:                   146   Pseudo R-squ. (CS):             0.1138
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------

In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Extraindo a tabela de resultados do modelo GLM para um DataFrame do Pandas
df_coef = pd.DataFrame({
    'variavel': modelo_nacional.params.index,
    'coeficiente': modelo_nacional.params.values,
    'erro_padrao': modelo_nacional.bse.values,
    'z_stat': modelo_nacional.tvalues.values,
        'p_valor': modelo_nacional.pvalues.values,
    'ci_inferior': modelo_nacional.conf_int()[0].values,
    'ci_superior': modelo_nacional.conf_int()[1].values
})

# Calculando a Razão de Chances (Odds Ratio / Exponencial do Coeficiente) para facilitar a interpretação
df_coef['razao_de_chances'] = np.exp(df_coef['coeficiente'])

# Ordenando por relevância estatística (menor p-valor)
df_coef = df_coef.sort_values(by='p_valor')

# 2. Definindo o diretório de destino na camada Gold do WSL
gold_path = "/home/haroldo/tcc_bacen_scr/data/processed/gold"
os.makedirs(gold_path, exist_ok=True)

# 3. Salvando em CSV e em Excel (para você abrir direto ou jogar no Power BI)
caminho_csv = os.path.join(gold_path, "coeficientes_modelo_glm.csv")
caminho_xlsx = os.path.join(gold_path, "coeficientes_modelo_glm.xlsx")

df_coef.to_csv(caminho_csv, index=False, encoding='utf-8-sig')
df_coef.to_excel(caminho_xlsx, index=False)

print(f"\n💾 Coeficientes exportados com sucesso!")
print(f"   - CSV: {caminho_csv}")
print(f"   - Excel: {caminho_xlsx}")